# Практическая работа: банковские счета на Python

**Тема:** ООП, инкапсуляция, наследование, полиморфизм, история операций, `datetime`.

**Цель:** реализовать `Account` и `CreditAccount` в соответствии с заданием и дополнить проект функциями, которые можно продемонстрировать преподавателю.

**Как пользоваться:** в Google Colab выполните **Среда выполнения → Выполнить все**. В каждой программной ячейке приведены подробные построчные комментарии. Комментарии начинаются с `#` и не влияют на работу программы.

### Что добавлено сверх обязательной части

1. **Полиморфизм**: единая логика снятия денег в `Account`, расширенная в `CreditAccount` через переопределение вспомогательных методов.
2. **Защита данных**: история возвращается в виде независимой глубокой копии, а ввод проверяется на `NaN`, бесконечность и логические значения.
3. **Аналитика**: количество успешных и неуспешных операций, обороты по депозитам и снятиям, число использований кредита.
4. **Удобный отчёт**: таблица операций с отдельной колонкой «Кредит использован».
5. **Автотесты**: тесты граничных случаев, наследования, времени, инкапсуляции и отсутствия превышения кредитного лимита.

> **Примечание об учебной модели.** По условию суммы хранятся как `float`. В реальных банковских системах вместо `float` обычно используют `Decimal` или целые числа минимальных денежных единиц во избежание ошибок двоичной арифметики.

## 1. Импорт библиотек

Все импорты входят в стандартную библиотеку Python. Для табличного отчёта в последнем разделе используется `pandas`, который уже установлен в Google Colab.

In [1]:
from copy import deepcopy  # Создаёт независимые копии вложенных структур данных.
from datetime import datetime  # Возвращает дату и время каждой операции.
import math  # Помогает отклонять NaN и бесконечные значения.
from numbers import Real  # Позволяет проверять вещественные и целые числа.
import unittest  # Запускает автоматические проверки логики программы.

## 2. Базовый класс `Account`

**Инкапсуляция:** `_balance` считается внутренним атрибутом. Доступ к его значению предоставляется через `get_balance()`.

**Структура истории:** каждый элемент — словарь с полями `type`, `amount`, `datetime`, `balance`, `status`. Дополнительное поле `credit_used` в кредитном счёте показывает, потребовала ли **конкретная успешная операция снятия** использования заёмных средств.

**Обработка ошибок:** некорректная сумма (`0`, отрицательное значение, строка, `NaN`) приводит к `ValueError` и попадает в историю со статусом `fail`. При нехватке средств `withdraw()` возвращает `False` и тоже сохраняет неудачную попытку.

In [2]:
class Account:  # Описываем общий тип банковского счёта.
    """Обычный банковский счёт с историей операций и аналитикой."""

    def __init__(self, account_holder, balance=0):  # Конструктор счёта.
        if not isinstance(account_holder, str):  # Проверяем тип имени владельца.
            raise TypeError("Имя владельца должно быть строкой.")  # Не принимаем иной тип.
        if not account_holder.strip():  # Проверяем, что имя не пустое.
            raise ValueError("Имя владельца не должно быть пустым.")  # Сообщаем об ошибке.
        self._check_number(balance, "Начальный баланс")  # Проверяем тип и конечность суммы.
        if balance < self._minimum_balance():  # Учитываем границу допустимого баланса.
            raise ValueError("Недопустимый начальный баланс.")  # Отклоняем слишком малый баланс.
        self.holder = account_holder.strip()  # Сохраняем имя без пробелов по краям.
        self._balance = float(balance)  # Храним баланс во внутреннем атрибуте.
        self.operations_history = []  # Создаём отдельный список истории для счёта.

    @staticmethod  # Метод не использует состояние объекта.
    def _check_number(value, field_name):  # Проверяем корректность числового значения.
        if isinstance(value, bool) or not isinstance(value, Real):  # Исключаем bool и нечисловые типы.
            raise ValueError(f"{field_name}: требуется число.")  # Возвращаем понятную ошибку.
        if not math.isfinite(value):  # Выявляем NaN, +inf и -inf.
            raise ValueError(f"{field_name}: число должно быть конечным.")  # Отклоняем их.

    def _minimum_balance(self):  # Точка расширения для дочернего класса.
        return 0.0  # У обычного счёта отрицательный остаток запрещён.

    def _operation_details(self, operation_type, amount, status, old_balance):  # Дополнительные поля.
        return {}  # Обычному счёту специальные поля не нужны.

    def _record_operation(self, operation_type, amount, status, old_balance):  # Записываем событие.
        entry = {  # Создаём одну структурированную запись истории.
            "type": operation_type,  # Тип: deposit или withdraw.
            "amount": amount,  # Запрошенная сумма, в том числе при отказе.
            "datetime": datetime.now(),  # Реальные дата и время попытки.
            "balance": self._balance,  # Остаток после операции или отказа.
            "status": status,  # Статус: success или fail.
        }  # Завершаем создание словаря.
        entry.update(  # Расширяем запись полями наследника, если они есть.
            self._operation_details(operation_type, amount, status, old_balance)
        )  # Заканчиваем добавление дополнительных полей.
        self.operations_history.append(entry)  # Добавляем запись в историю.

    def _validate_operation_amount(self, amount, operation_type):  # Проверяем сумму операции.
        try:  # При ошибке записываем саму неудачную попытку.
            self._check_number(amount, "Сумма операции")  # Проверяем тип и конечность.
            if amount <= 0:  # Сумма депозита и снятия должна быть строго положительной.
                raise ValueError("Сумма операции должна быть больше нуля.")  # Отклоняем её.
        except ValueError:  # Перехватываем ошибку проверки.
            self._record_operation(operation_type, amount, "fail", self._balance)  # Логируем отказ.
            raise  # Передаём исходную ошибку вызывающему коду.

    def deposit(self, amount):  # Метод пополнения баланса.
        self._validate_operation_amount(amount, "deposit")  # Проверяем правильность суммы.
        old_balance = self._balance  # Запоминаем прежний остаток для журнала.
        self._balance += float(amount)  # Увеличиваем баланс на сумму пополнения.
        self._record_operation("deposit", amount, "success", old_balance)  # Логируем успех.
        return True  # Подтверждаем выполнение операции.

    def withdraw(self, amount):  # Метод снятия денег.
        self._validate_operation_amount(amount, "withdraw")  # Проверяем сумму снятия.
        old_balance = self._balance  # Сохраняем баланс до снятия.
        if self._balance - amount < self._minimum_balance():  # Проверяем нижнюю границу.
            self._record_operation("withdraw", amount, "fail", old_balance)  # Логируем отказ.
            return False  # Возвращаем признак недостатка средств.
        self._balance -= float(amount)  # Списываем средства при допустимом остатке.
        self._record_operation("withdraw", amount, "success", old_balance)  # Логируем успех.
        return True  # Подтверждаем снятие.

    def get_balance(self):  # Геттер текущего баланса.
        return self._balance  # Возвращаем число, не меняя объект.

    def get_history(self):  # Безопасный геттер журнала.
        return deepcopy(self.operations_history)  # Не даём менять историю через результат.

    def get_summary(self):  # Дополнительная аналитика для оценки выше базовой.
        history = self.get_history()  # Получаем защищённую копию записей.
        successful = [item for item in history if item["status"] == "success"]  # Успехи.
        failed = [item for item in history if item["status"] == "fail"]  # Отказы.
        deposits = sum(  # Считаем общую сумму успешных пополнений.
            item["amount"] for item in successful if item["type"] == "deposit"
        )  # Итог по пополнениям.
        withdrawals = sum(  # Считаем общую сумму успешных снятий.
            item["amount"] for item in successful if item["type"] == "withdraw"
        )  # Итог по снятиям.
        credit_operations = sum(  # Считаем снятия, использовавшие кредитные деньги.
            bool(item.get("credit_used", False)) for item in successful
        )  # Итог по кредитным операциям.
        return {  # Возвращаем структурированный аналитический отчёт.
            "holder": self.holder,  # Владелец счёта.
            "balance": self._balance,  # Текущий остаток.
            "total_operations": len(history),  # Все попытки, включая отклонённые.
            "successful_operations": len(successful),  # Успешные операции.
            "failed_operations": len(failed),  # Отказанные операции.
            "total_deposits": deposits,  # Успешный оборот пополнений.
            "total_withdrawals": withdrawals,  # Успешный оборот снятий.
            "credit_operations": credit_operations,  # Количество снятий с использованием кредита.
        }  # Завершаем словарь отчёта.

## 3. Дочерний класс `CreditAccount`

Здесь показано **наследование** (`CreditAccount(Account)`) и **полиморфизм**: базовый `withdraw()` автоматически учитывает кредитный лимит, потому что обращается к переопределённому `_minimum_balance()`.

В отличие от копирования кода `withdraw()`, такой подход сохраняет единый алгоритм снятия и облегчает сопровождение проекта. `super().__init__()` запускает конструктор базового класса.

**Договорённость о `credit_used`:** `True` при успешном снятии, если итоговый остаток отрицателен; `False` при пополнении или неуспешном снятии. Баланс и доступный кредит записаны отдельно.

In [3]:
class CreditAccount(Account):  # Наследуем все основные возможности Account.
    """Счёт, допускающий отрицательный баланс в пределах кредитного лимита."""

    def __init__(self, account_holder, balance=0, credit_limit=0):  # Конструктор кредитного счёта.
        self._check_number(credit_limit, "Кредитный лимит")  # Проверяем число и конечность.
        if credit_limit < 0:  # Кредитный лимит не может быть отрицательным.
            raise ValueError("Кредитный лимит не может быть отрицательным.")  # Отклоняем его.
        self.credit_limit = float(credit_limit)  # Сохраняем лимит до вызова super().
        super().__init__(account_holder, balance)  # Вызываем базовый конструктор с проверками.

    def _minimum_balance(self):  # Переопределяем нижнюю границу остатка.
        return -self.credit_limit  # Разрешаем долг не больше кредитного лимита.

    def get_available_credit(self):  # Показываем доступные средства по условию задания.
        return self._balance + self.credit_limit  # Остаток плюс лимит.

    def _operation_details(self, operation_type, amount, status, old_balance):  # Расширяем журнал.
        used = (  # Определяем, задействовала ли операция кредит.
            operation_type == "withdraw"  # Кредит используется только при снятии.
            and status == "success"  # Только успешное снятие расходует средства.
            and self._balance < 0  # Отрицательный остаток означает кредитные деньги.
        )  # Получаем логическое значение.
        return {  # Отдаём дополнительные сведения базовому журналу.
            "credit_used": used,  # Отмечаем факт использования кредита.
            "available_credit": self.get_available_credit(),  # Фиксируем ресурс после операции.
        }  # Завершаем словарь дополнительных полей.

## 4. Демонстрация базовых операций

Одна операция намеренно завершается отказом: важно показать, что она **не изменяет баланс**, но появляется в журнале.

In [4]:
account = Account("Анна Смирнова", 1000)  # Создаём обычный счёт с 1000 единицами.
print("Начальный баланс:", account.get_balance())  # Проверяем сумму после создания.
print("Пополнение:", account.deposit(500))  # Увеличиваем баланс до 1500.
print("Снятие:", account.withdraw(300))  # Снимаем 300, остаётся 1200.
print("Снятие сверх остатка:", account.withdraw(2000))  # Получаем False без списания.
print("Конечный баланс:", account.get_balance())  # Ожидаем 1200.
print("Аналитика:", account.get_summary())  # Просматриваем общий отчёт.

Начальный баланс: 1000.0
Пополнение: True
Снятие: True
Снятие сверх остатка: False
Конечный баланс: 1200.0
Аналитика: {'holder': 'Анна Смирнова', 'balance': 1200.0, 'total_operations': 3, 'successful_operations': 2, 'failed_operations': 1, 'total_deposits': 500, 'total_withdrawals': 300, 'credit_operations': 0}


## 5. Демонстрация кредитного счёта

Покажем переход в отрицательный баланс, частичное погашение долга и отклонение снятия сверх лимита.

In [5]:
credit_account = CreditAccount("Иван Петров", 200, 1000)  # 200 своих и 1000 заёмных.
print("Стартовый доступный кредит:", credit_account.get_available_credit())  # 1200.
print("Снятие 700:", credit_account.withdraw(700))  # Остаток становится -500.
print("Баланс после снятия:", credit_account.get_balance())  # Проверяем -500.
print("Осталось доступно:", credit_account.get_available_credit())  # Получаем 500.
print("Снятие 600:", credit_account.withdraw(600))  # Получаем отказ: превышен лимит.
print("Погашение 250:", credit_account.deposit(250))  # Остаток становится -250.
print("Осталось доступно:", credit_account.get_available_credit())  # Получаем 750.
print("Аналитика:", credit_account.get_summary())  # Смотрим статистику событий.

Стартовый доступный кредит: 1200.0
Снятие 700: True
Баланс после снятия: -500.0
Осталось доступно: 500.0
Снятие 600: False
Погашение 250: True
Осталось доступно: 750.0
Аналитика: {'holder': 'Иван Петров', 'balance': -250.0, 'total_operations': 3, 'successful_operations': 2, 'failed_operations': 1, 'total_deposits': 250, 'total_withdrawals': 700, 'credit_operations': 1}


## 6. Наглядная таблица операций

Преобразуем журналы в `DataFrame`. Это **не изменяет историю**, а только создаёт удобную для проверки таблицу. В Google Colab `pandas` уже доступен.

In [6]:
import pandas as pd  # Подключаем библиотеку для удобного табличного вывода.


def show_history(bank_account):  # Функция подходит для обоих видов счёта.
    rows = bank_account.get_history()  # Получаем безопасную копию журнала.
    table = pd.DataFrame(rows)  # Превращаем записи-словари в таблицу.
    if not table.empty:  # Форматируем время только при наличии строк.
        table["datetime"] = table["datetime"].dt.strftime(  # Меняем вид времени.
            "%d.%m.%Y %H:%M:%S"  # Формат для человека: день, месяц, год и часы.
        )  # Завершаем форматирование колонки.
    return table  # Возвращаем таблицу, удобную для просмотра.


print("История обычного счёта:")  # Подписываем первую таблицу.
display(show_history(account))  # Отображаем журнал обычного счёта в Colab.
print("История кредитного счёта:")  # Подписываем вторую таблицу.
display(show_history(credit_account))  # Отображаем расширенную историю кредита.

История обычного счёта:


,type,amount,datetime,balance,status
0,deposit,500,04.11.2026 16:12:25,1500.0,success
1,withdraw,300,04.11.2026 16:12:25,1200.0,success
2,withdraw,2000,04.11.2026 16:12:25,1200.0,fail


История кредитного счёта:


,type,amount,datetime,balance,status,credit_used,available_credit
0,withdraw,700,04.11.2026 16:12:25,-500.0,success,True,500.0
1,withdraw,600,04.11.2026 16:12:25,-500.0,fail,False,500.0
2,deposit,250,04.11.2026 16:12:25,-250.0,success,False,750.0


## 7. Автоматические тесты `unittest`

Тесты дают преподавателю возможность **воспроизвести доказательство корректности**: они проверяют не только типичные действия, но и ошибки ввода, граничные случаи, дату/время, наследование и защиту истории.

Каждая проверка снабжена пояснением; новые тесты можно добавить по аналогии.

In [7]:
class TestBankAccounts(unittest.TestCase):  # Объединяем проверки в один тестовый набор.
    """Проверки требований задания и дополнительных возможностей."""

    def test_default_balance(self):  # Проверяем баланс по умолчанию.
        account = Account("Тест")  # Создаём счёт без начальной суммы.
        self.assertEqual(account.get_balance(), 0.0)  # Баланс должен равняться нулю.

    def test_negative_initial_balance(self):  # Проверяем запрет обычного овердрафта.
        with self.assertRaises(ValueError):  # Ожидаем ошибку создания.
            Account("Тест", -1)  # Передаём недопустимый остаток.

    def test_deposit_and_withdraw(self):  # Проверяем успешные операции.
        account = Account("Тест", 100)  # Начинаем со 100.
        self.assertTrue(account.deposit(50))  # Успешное пополнение возвращает True.
        self.assertTrue(account.withdraw(20))  # Успешное снятие возвращает True.
        self.assertEqual(account.get_balance(), 130)  # Остаток должен равняться 130.

    def test_insufficient_funds_logged(self):  # Проверяем отказ и историю.
        account = Account("Тест", 10)  # На балансе только 10.
        self.assertFalse(account.withdraw(11))  # Снятие 11 невозможно.
        self.assertEqual(account.get_balance(), 10)  # Баланс не меняется.
        self.assertEqual(account.get_history()[0]["status"], "fail")  # Отказ записан.

    def test_invalid_amount_logged(self):  # Проверяем ошибочную сумму.
        account = Account("Тест", 100)  # Создаём счёт.
        with self.assertRaises(ValueError):  # Ожидаем проверяемое исключение.
            account.deposit(-2)  # Отрицательное пополнение недопустимо.
        self.assertEqual(account.get_history()[0]["status"], "fail")  # Попытка записана.
        self.assertEqual(account.get_balance(), 100)  # Остаток не изменился.

    def test_zero_nan_and_infinity_rejected(self):  # Проверяем особые значения.
        account = Account("Тест", 100)  # Создаём тестовый объект.
        for amount in (0, float("nan"), float("inf"), True):  # Перебираем ошибки.
            with self.subTest(amount=amount):  # Выделяем случай в отдельную проверку.
                with self.assertRaises(ValueError):  # Каждое значение должно отклоняться.
                    account.withdraw(amount)  # Пробуем некорректное снятие.
        self.assertEqual(account.get_balance(), 100)  # Все попытки оставили баланс прежним.
        self.assertEqual(len(account.get_history()), 4)  # Все четыре отказа журналируются.

    def test_datetime_saved(self):  # Проверяем регистрацию времени.
        account = Account("Тест")  # Создаём пустой счёт.
        account.deposit(5)  # Выполняем одну операцию.
        self.assertIsInstance(account.get_history()[0]["datetime"], datetime)  # Это datetime.

    def test_history_copy_is_independent(self):  # Проверяем защиту истории от изменений.
        account = Account("Тест")  # Создаём счёт.
        account.deposit(10)  # Добавляем запись.
        history_copy = account.get_history()  # Получаем копию данных.
        history_copy[0]["status"] = "fail"  # Намеренно портим копию.
        self.assertEqual(account.get_history()[0]["status"], "success")  # Оригинал цел.

    def test_credit_negative_initial_balance(self):  # Проверяем кредитную инициализацию.
        account = CreditAccount("Тест", -200, 500)  # Допустимый стартовый долг.
        self.assertEqual(account.get_available_credit(), 300)  # Доступно ещё 300.
        with self.assertRaises(ValueError):  # Стартовый долг свыше лимита запрещён.
            CreditAccount("Тест", -501, 500)  # Пробуем нарушить ограничение.

    def test_credit_limit_boundary(self):  # Проверяем точную границу кредита.
        account = CreditAccount("Тест", 100, 200)  # Всего доступно 300.
        self.assertTrue(account.withdraw(300))  # Ровно до -200 снимать можно.
        self.assertEqual(account.get_balance(), -200)  # Достигнута граница.
        self.assertEqual(account.get_available_credit(), 0)  # Ресурс исчерпан.
        self.assertFalse(account.withdraw(1))  # Ещё один рубль списать нельзя.
        self.assertEqual(account.get_balance(), -200)  # Баланс остался на границе.

    def test_credit_used_and_repayment(self):  # Проверяем флаги кредита и погашение.
        account = CreditAccount("Тест", 20, 100)  # Есть 20 собственных средств.
        account.withdraw(50)  # Расходуем 30 кредитных средств.
        account.deposit(10)  # Частично погашаем задолженность.
        account.withdraw(200)  # Неудачная попытка превысить лимит.
        history = account.get_history()  # Получаем журнал.
        self.assertTrue(history[0]["credit_used"])  # Первое снятие использовало кредит.
        self.assertFalse(history[1]["credit_used"])  # Пополнение кредит не расходует.
        self.assertFalse(history[2]["credit_used"])  # Отказ не использует кредит.
        self.assertEqual(account.get_balance(), -20)  # Долг уменьшился до 20.

    def test_summary(self):  # Проверяем дополнительную аналитику.
        account = Account("Тест", 100)  # Стартуем со 100.
        account.deposit(40)  # Успешно пополняем.
        account.withdraw(10)  # Успешно снимаем.
        account.withdraw(1000)  # Получаем один отказ.
        summary = account.get_summary()  # Рассчитываем статистику.
        self.assertEqual(summary["total_operations"], 3)  # Всего три события.
        self.assertEqual(summary["failed_operations"], 1)  # Одно событие — отказ.
        self.assertEqual(summary["total_deposits"], 40)  # Оборот пополнений равен 40.
        self.assertEqual(summary["total_withdrawals"], 10)  # Оборот снятий равен 10.


suite = unittest.defaultTestLoader.loadTestsFromTestCase(  # Загружаем все тесты класса.
    TestBankAccounts  # Указываем класс, содержащий сценарии.
)  # Набор тестов готов.
result = unittest.TextTestRunner(verbosity=2).run(suite)  # Запускаем с подробным отчётом.
assert result.wasSuccessful(), "Есть ошибки: проверьте результаты тестов."  # Контроль итога.

test_credit_limit_boundary (__main__.TestBankAccounts.test_credit_limit_boundary) ... 

ok


test_credit_negative_initial_balance (__main__.TestBankAccounts.test_credit_negative_initial_balance) ... 

ok


test_credit_used_and_repayment (__main__.TestBankAccounts.test_credit_used_and_repayment) ... 

ok


... (15 outputs left out)

ok


test_zero_nan_and_infinity_rejected (__main__.TestBankAccounts.test_zero_nan_and_infinity_rejected) ... 

ok


----------------------------------------------------------------------
Ran 12 tests in 0.017s

OK


## 8. Что можно рассказать преподавателю при защите

**1. Почему поля разделены?** `holder` хранит имя, `_balance` — внутреннее состояние, `operations_history` — журнал событий.

**2. Почему история хранит словари?** У каждого события есть тип, сумма, дата и время, остаток, статус. У кредитного счёта есть дополнительные поля, и их легко читать по ключам.

**3. В чём наследование?** `CreditAccount` использует конструктор и методы `Account`, переопределяя только допустимую нижнюю границу и дополнительные данные журнала.

**4. В чём полиморфизм?** Одна реализация `withdraw()` в родительском классе ведёт себя по-разному для обычного и кредитного счёта благодаря `_minimum_balance()`.

**5. Почему `get_history()` возвращает копию?** Чтобы внешняя программа не могла подменить уже зарегистрированный результат операции простым изменением возвращённого списка.

**6. Какие улучшения сверх задания?** Проверка чисел `NaN` / `inf` / `bool`, копирование истории, статистика операций, табличное представление, 12 автоматических тестов.

**7. Что улучшить для реального банка?** Перейти с `float` на `Decimal`, использовать UTC и часовые пояса, хранить неизменяемый аудит в БД и добавить аутентификацию с транзакционной обработкой.

---

### Как сдать

1. Загрузите файл `.ipynb` в **Google Colab** и выберите **Среда выполнения → Выполнить все**.
2. Убедитесь, что тесты завершились успешно.
3. Выберите **Поделиться → Общий доступ → Все, у кого есть ссылка → Читатель** (если правила курса это допускают).
4. Откройте ссылку в режиме инкогнито, чтобы проверить доступ без входа в ваш аккаунт.
5. Прикрепите преподавателю ссылку на блокнот (не ссылку на локальный файл).